In [1]:
import os
import gc
import random
import warnings
import itertools

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score
)

# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")

# CPU 執行緒可依電腦調整
tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)


# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")


# =========================================================
# C. 反轉反向指標（LLaMA）
# 分數越高 -> 越漂綠 / 風險越高
# =========================================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)


# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]


# =========================================================
# E. M6 特徵欄位（LLaMA）
# =========================================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

m6_cols = semantic_cols + lexical_cols + financial_cols

missing_cols = [c for c in (m6_cols + ["label", "Company"]) if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")


# =========================================================
# F. CV 設定
# Outer: 真正泛化評估
# Inner: 調參 + OOF threshold
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)


# =========================================================
# G. 前處理
# 連續欄位: median + standardize
# lexical欄位: most_frequent
# =========================================================
def preprocess_train_valid_test(train_df, valid_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_valid_cont = cont_imputer.transform(valid_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_valid_cont = scaler.transform(X_valid_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_cont = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_valid_lex = lex_imputer.transform(valid_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_lex = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_valid = np.concatenate([X_valid_cont, X_valid_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_valid, X_test


def preprocess_train_test_only(train_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_test


# =========================================================
# H. 小工具
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


def fast_predict(model, X, batch_size=256):
    X = np.asarray(X, dtype=np.float32)
    return model.predict(X, batch_size=batch_size, verbose=0).ravel()


def find_best_threshold(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.01)

    best_threshold = 0.50
    best_score = -1.0

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = th

    return float(best_threshold), float(best_score)


def make_class_weight(y_train):
    classes = np.unique(y_train)
    counts = np.bincount(y_train)
    total = len(y_train)
    class_weight = {}
    for cls in classes:
        class_weight[int(cls)] = total / (len(classes) * counts[cls])
    return class_weight


# =========================================================
# I. FT-Transformer 簡化版元件
# =========================================================
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    """
    將每個 numerical/binary feature 當作一個 token
    x: (batch, num_features)
    output: (batch, num_features, d_token)
    """
    def __init__(self, num_features, d_token, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.d_token = d_token

    def build(self, input_shape):
        self.weight = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True,
            name="feature_weight"
        )
        self.bias = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="zeros",
            trainable=True,
            name="feature_bias"
        )
        self.col_embedding = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True,
            name="column_embedding"
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.expand_dims(x, axis=-1)                     # (B, F, 1)
        tokens = x * self.weight + self.bias              # (B, F, D)
        tokens = tokens + self.col_embedding              # column identity
        return tokens


class CLSLayer(tf.keras.layers.Layer):
    def __init__(self, d_token, **kwargs):
        super().__init__(**kwargs)
        self.d_token = d_token

    def build(self, input_shape):
        self.cls = self.add_weight(
            shape=(1, 1, self.d_token),
            initializer="zeros",
            trainable=True,
            name="cls_token"
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls = tf.repeat(self.cls, repeats=batch_size, axis=0)
        return tf.concat([cls, x], axis=1)


def transformer_block(x, d_token, num_heads, ff_dim, dropout_rate):
    # PreNorm
    h = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_token // num_heads if d_token >= num_heads else 1,
        dropout=dropout_rate
    )(h, h)
    h = tf.keras.layers.Dropout(dropout_rate)(h)
    x = tf.keras.layers.Add()([x, h])

    h2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h2 = tf.keras.layers.Dense(ff_dim, activation="relu")(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    h2 = tf.keras.layers.Dense(d_token)(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    x = tf.keras.layers.Add()([x, h2])

    return x


def build_ft_transformer(
    num_features,
    d_token=16,
    num_blocks=2,
    num_heads=4,
    ff_dim=32,
    dropout_rate=0.2,
    mlp_hidden=16,
    learning_rate=1e-3
):
    inputs = tf.keras.Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    x = NumericalFeatureTokenizer(num_features=num_features, d_token=d_token)(inputs)
    x = CLSLayer(d_token=d_token)(x)

    for _ in range(num_blocks):
        x = transformer_block(
            x=x,
            d_token=d_token,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate
        )

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    cls_token = x[:, 0, :]   # 取 cls token

    x = tf.keras.layers.Dense(mlp_hidden, activation="relu")(cls_token)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# =========================================================
# J. 內層評估單一參數組
# Nested inner CV 用來選參數
# =========================================================
def evaluate_param_on_inner_cv(X_df, y, groups, cont_cols, lex_cols, params):
    inner_scores = []

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,   # 這裡 test 不用，只是共用函式
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )
        X_inner_val = X_inner_val.astype(np.float32)

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            d_token=params["d_token"],
            num_blocks=params["num_blocks"],
            num_heads=params["num_heads"],
            ff_dim=params["ff_dim"],
            dropout_rate=params["dropout_rate"],
            mlp_hidden=params["mlp_hidden"],
            learning_rate=params["learning_rate"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )

        class_weight = make_class_weight(y_inner_train.values)

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        val_pred = (val_prob >= 0.5).astype(int)
        val_f1 = f1_score(y_inner_val, val_pred, zero_division=0)
        inner_scores.append(val_f1)

        del model
        cleanup_tf()

    return float(np.mean(inner_scores))


# =========================================================
# K. 用最佳參數在 outer-train 內做 OOF thresholding
# =========================================================
def get_oof_probs_for_threshold(X_df, y, groups, cont_cols, lex_cols, best_params):
    oof_prob = pd.Series(index=X_df.index, dtype=float)

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )
        X_inner_val = X_inner_val.astype(np.float32)

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            d_token=best_params["d_token"],
            num_blocks=best_params["num_blocks"],
            num_heads=best_params["num_heads"],
            ff_dim=best_params["ff_dim"],
            dropout_rate=best_params["dropout_rate"],
            mlp_hidden=best_params["mlp_hidden"],
            learning_rate=best_params["learning_rate"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )

        class_weight = make_class_weight(y_inner_train.values)

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=best_params["epochs"],
            batch_size=best_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        oof_prob.loc[inner_val_df.index] = val_prob

        del model
        cleanup_tf()

    if oof_prob.isna().any():
        raise ValueError("OOF probability 有缺值，請檢查 inner_cv 切分")

    return oof_prob


# =========================================================
# L. outer evaluation（只跑 M6）
# =========================================================
def evaluate_ft_transformer_m6(df, y, groups, selected_cols):
    fold_metrics = []
    best_params_per_fold = []

    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]

    # 簡化版參數空間：不要太大，先測可行性
    param_grid = {
        "d_token": [16, 24],
        "num_blocks": [2],
        "num_heads": [4],
        "ff_dim": [32, 64],
        "dropout_rate": [0.2],
        "mlp_hidden": [16],
        "learning_rate": [1e-3],
        "batch_size": [32],
        "epochs": [25]
    }

    param_list = list(itertools.product(
        param_grid["d_token"],
        param_grid["num_blocks"],
        param_grid["num_heads"],
        param_grid["ff_dim"],
        param_grid["dropout_rate"],
        param_grid["mlp_hidden"],
        param_grid["learning_rate"],
        param_grid["batch_size"],
        param_grid["epochs"]
    ))

    param_dicts = [
        {
            "d_token": p[0],
            "num_blocks": p[1],
            "num_heads": p[2],
            "ff_dim": p[3],
            "dropout_rate": p[4],
            "mlp_hidden": p[5],
            "learning_rate": p[6],
            "batch_size": p[7],
            "epochs": p[8]
        }
        for p in param_list
    ]

    print("\n" + "=" * 100)
    print("Running FT-Transformer (simplified) on M6 only")
    print("=" * 100)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[M6] Outer Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_train_df = train_df[selected_cols].copy()
        X_test_df = test_df[selected_cols].copy()

        # -----------------------------
        # 1) inner nested validation: 選最佳參數
        # -----------------------------
        best_score = -1.0
        best_params = None

        for params in param_dicts:
            mean_inner_f1 = evaluate_param_on_inner_cv(
                X_df=X_train_df,
                y=y_train,
                groups=groups_train,
                cont_cols=cont_cols,
                lex_cols=lex_cols,
                params=params
            )

            if mean_inner_f1 > best_score:
                best_score = mean_inner_f1
                best_params = params.copy()

        best_params_per_fold.append(best_params)
        print(f"[M6] Fold {fold_idx} Best Params: {best_params}")
        print(f"[M6] Fold {fold_idx} Best Inner F1 (0.5 threshold): {best_score:.4f}")

        # -----------------------------
        # 2) outer-train 內做 OOF thresholding
        # -----------------------------
        train_oof_prob = get_oof_probs_for_threshold(
            X_df=X_train_df,
            y=y_train,
            groups=groups_train,
            cont_cols=cont_cols,
            lex_cols=lex_cols,
            best_params=best_params
        )

        best_threshold, best_oof_f1 = find_best_threshold(
            y_true=y_train.loc[train_oof_prob.index].values,
            y_prob=train_oof_prob.values
        )

        # -----------------------------
        # 3) 用最佳參數重訓 outer train，評估 outer test
        # -----------------------------
        X_train_full, X_test = preprocess_train_test_only(
            train_df=X_train_df,
            test_df=X_test_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_train_full.shape[1],
            d_token=best_params["d_token"],
            num_blocks=best_params["num_blocks"],
            num_heads=best_params["num_heads"],
            ff_dim=best_params["ff_dim"],
            dropout_rate=best_params["dropout_rate"],
            mlp_hidden=best_params["mlp_hidden"],
            learning_rate=best_params["learning_rate"]
        )

        # 再切一小塊 validation 給 early stopping
        val_ratio = 0.15
        n_train = len(X_train_full)
        val_size = max(1, int(n_train * val_ratio))

        X_tr = X_train_full[:-val_size]
        X_val = X_train_full[-val_size:]
        y_tr = y_train.iloc[:-val_size].values
        y_val = y_train.iloc[-val_size:].values

        class_weight = make_class_weight(y_tr)

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )

        model.fit(
            X_tr,
            y_tr,
            validation_data=(X_val, y_val),
            epochs=best_params["epochs"],
            batch_size=best_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob),
            "best_threshold": best_threshold,
            "oof_best_f1": best_oof_f1
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[M6] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f} | "
            f"PR_AUC={fold_result['average_precision']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "FT-Transformer (simplified)",
        "Feature_Set": "M6: Semantic + Lexical + Financial",
        "Config": str(best_params_per_fold),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(ddof=1),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(ddof=1),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(ddof=1),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(ddof=1),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(ddof=1),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(ddof=1),

        "Mean_Best_Threshold": metrics_df["best_threshold"].mean(),
        "Threshold_std": metrics_df["best_threshold"].std(ddof=1),

        "Mean_OOF_Best_F1": metrics_df["oof_best_f1"].mean(),
        "OOF_Best_F1_std": metrics_df["oof_best_f1"].std(ddof=1)
    }

    return summary, metrics_df


# =========================================================
# M. 執行
# =========================================================
summary_result, fold_result_df = evaluate_ft_transformer_m6(
    df=df,
    y=y,
    groups=groups_all,
    selected_cols=m6_cols
)

results_df = pd.DataFrame([summary_result])
folds_df = fold_result_df.copy()

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== Summary Results =====")
print(results_df)

results_df.to_csv(
    "llama_ft_transformer_simplified_M6_nested_groupcv_oof_threshold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "llama_ft_transformer_simplified_M6_nested_groupcv_oof_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n結果已輸出：")
print("1. llama_ft_transformer_simplified_M6_nested_groupcv_oof_threshold_results.csv")
print("2. llama_ft_transformer_simplified_M6_nested_groupcv_oof_threshold_fold_results.csv")

I0000 00:00:1774943030.007079 1344416 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774943030.008926 1344416 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774943030.061175 1344416 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774943031.333364 1344416 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE


Running FT-Transformer (simplified) on M6 only

[M6] Outer Fold 1 started


E0000 00:00:1774943032.480179 1344416 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
E0000 00:00:1774943035.731038 1344416 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


[M6] Fold 1 Best Params: {'d_token': 16, 'num_blocks': 2, 'num_heads': 4, 'ff_dim': 32, 'dropout_rate': 0.2, 'mlp_hidden': 16, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 25}
[M6] Fold 1 Best Inner F1 (0.5 threshold): 0.3806
[M6] Fold 1 done | Threshold=0.41 | F1=0.1538 | ROC_AUC=0.8295 | PR_AUC=0.6926

[M6] Outer Fold 2 started
[M6] Fold 2 Best Params: {'d_token': 16, 'num_blocks': 2, 'num_heads': 4, 'ff_dim': 32, 'dropout_rate': 0.2, 'mlp_hidden': 16, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 25}
[M6] Fold 2 Best Inner F1 (0.5 threshold): 0.4286
[M6] Fold 2 done | Threshold=0.73 | F1=0.0000 | ROC_AUC=0.8627 | PR_AUC=0.3484

[M6] Outer Fold 3 started
[M6] Fold 3 Best Params: {'d_token': 24, 'num_blocks': 2, 'num_heads': 4, 'ff_dim': 64, 'dropout_rate': 0.2, 'mlp_hidden': 16, 'learning_rate': 0.001, 'batch_size': 32, 'epochs': 25}
[M6] Fold 3 Best Inner F1 (0.5 threshold): 0.6556
[M6] Fold 3 done | Threshold=0.37 | F1=0.3000 | ROC_AUC=0.7568 | PR_AUC=0.3281

[M6] Oute

In [ ]:
import os
import gc
import random
import warnings
import itertools

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score
)
from tensorflow.keras import regularizers

# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")

tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)

# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# =========================================================
# C. 反轉反向指標（LLaMA）
# 分數越高 -> 越漂綠 / 風險越高
# =========================================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]

# =========================================================
# E. M6 特徵欄位（LLaMA）
# =========================================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

m6_cols = semantic_cols + lexical_cols + financial_cols

missing_cols = [c for c in (m6_cols + ["label", "Company"]) if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =========================================================
# F. CV 設定
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
final_val_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 999)

# =========================================================
# G. 前處理
# =========================================================
def preprocess_train_valid_test(train_df, valid_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_valid_cont = cont_imputer.transform(valid_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_valid_cont = scaler.transform(X_valid_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_cont = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_valid_lex = lex_imputer.transform(valid_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_lex = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_valid = np.concatenate([X_valid_cont, X_valid_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_valid, X_test


def preprocess_train_test_only(train_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_test

# =========================================================
# H. 小工具
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


def fast_predict(model, X, batch_size=256):
    X = np.asarray(X, dtype=np.float32)
    return model.predict(X, batch_size=batch_size, verbose=0).ravel()


def find_best_threshold(y_true, y_prob, thresholds=None):
    # 強化版：縮小搜尋範圍，避免 threshold 飆到 0.8 以上
    if thresholds is None:
        thresholds = np.arange(0.20, 0.61, 0.02)

    best_threshold = 0.50
    best_score = -1.0

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = th

    return float(best_threshold), float(best_score)


def make_class_weight(y_train, positive_multiplier=1.8):
    counts = np.bincount(y_train)
    total = len(y_train)
    n_classes = len(np.unique(y_train))

    class_weight = {}
    for cls in np.unique(y_train):
        class_weight[int(cls)] = total / (n_classes * counts[cls])

    # 額外強化正類
    if 1 in class_weight:
        class_weight[1] *= positive_multiplier

    return class_weight


def get_group_aware_train_val_split(X_df, y_series, groups_series, splitter):
    for tr_idx, val_idx in splitter.split(X_df, y_series, groups=groups_series):
        return tr_idx, val_idx
    raise ValueError("無法建立 final train/val split")

# =========================================================
# I. FT-Transformer 強化版元件
# =========================================================
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    """
    每個 numerical / binary feature 視為一個 token
    x: (batch, num_features)
    output: (batch, num_features, d_token)
    """
    def __init__(self, num_features, d_token, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.d_token = d_token

    def build(self, input_shape):
        self.weight = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True,
            name="feature_weight"
        )
        self.bias = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="zeros",
            trainable=True,
            name="feature_bias"
        )
        self.col_embedding = self.add_weight(
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True,
            name="column_embedding"
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.expand_dims(x, axis=-1)          # (B, F, 1)
        tokens = x * self.weight + self.bias    # (B, F, D)
        tokens = tokens + self.col_embedding
        return tokens


class CLSLayer(tf.keras.layers.Layer):
    def __init__(self, d_token, **kwargs):
        super().__init__(**kwargs)
        self.d_token = d_token

    def build(self, input_shape):
        self.cls = self.add_weight(
            shape=(1, 1, self.d_token),
            initializer="zeros",
            trainable=True,
            name="cls_token"
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls = tf.repeat(self.cls, repeats=batch_size, axis=0)
        return tf.concat([cls, x], axis=1)


def transformer_block(x, d_token, num_heads, ff_dim, dropout_rate, l2_reg):
    h = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_token // num_heads),
        dropout=dropout_rate
    )(h, h)
    h = tf.keras.layers.Dropout(dropout_rate)(h)
    x = tf.keras.layers.Add()([x, h])

    h2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h2 = tf.keras.layers.Dense(
        ff_dim,
        activation="gelu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    h2 = tf.keras.layers.Dense(
        d_token,
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    x = tf.keras.layers.Add()([x, h2])

    return x


def build_ft_transformer(
    num_features,
    d_token=32,
    num_blocks=2,
    num_heads=4,
    ff_dim=64,
    dropout_rate=0.2,
    feature_dropout_rate=0.05,
    mlp_hidden=32,
    learning_rate=1e-3,
    l2_reg=1e-5
):
    inputs = tf.keras.Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    # feature dropout：對小資料表格可增加穩定性
    x0 = tf.keras.layers.Dropout(feature_dropout_rate)(inputs)

    x = NumericalFeatureTokenizer(num_features=num_features, d_token=d_token)(x0)
    x = CLSLayer(d_token=d_token)(x)

    for _ in range(num_blocks):
        x = transformer_block(
            x=x,
            d_token=d_token,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate,
            l2_reg=l2_reg
        )

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    cls_token = x[:, 0, :]

    x = tf.keras.layers.Dense(
        mlp_hidden,
        activation="gelu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(cls_token)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# =========================================================
# J. inner CV 評估單一參數組
# =========================================================
def evaluate_param_on_inner_cv(X_df, y, groups, cont_cols, lex_cols, params):
    inner_scores = []

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            d_token=params["d_token"],
            num_blocks=params["num_blocks"],
            num_heads=params["num_heads"],
            ff_dim=params["ff_dim"],
            dropout_rate=params["dropout_rate"],
            feature_dropout_rate=params["feature_dropout_rate"],
            mlp_hidden=params["mlp_hidden"],
            learning_rate=params["learning_rate"],
            l2_reg=params["l2_reg"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        class_weight = make_class_weight(
            y_inner_train.values,
            positive_multiplier=params["positive_multiplier"]
        )

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        # inner 搜參數時仍先固定 0.5，比較公平
        val_pred = (val_prob >= 0.5).astype(int)
        val_f1 = f1_score(y_inner_val, val_pred, zero_division=0)
        inner_scores.append(val_f1)

        del model
        cleanup_tf()

    return float(np.mean(inner_scores))

# =========================================================
# K. 用最佳參數在 outer-train 內做 OOF thresholding
# =========================================================
def get_oof_probs_for_threshold(X_df, y, groups, cont_cols, lex_cols, best_params):
    oof_prob = pd.Series(index=X_df.index, dtype=float)

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            d_token=best_params["d_token"],
            num_blocks=best_params["num_blocks"],
            num_heads=best_params["num_heads"],
            ff_dim=best_params["ff_dim"],
            dropout_rate=best_params["dropout_rate"],
            feature_dropout_rate=best_params["feature_dropout_rate"],
            mlp_hidden=best_params["mlp_hidden"],
            learning_rate=best_params["learning_rate"],
            l2_reg=best_params["l2_reg"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        class_weight = make_class_weight(
            y_inner_train.values,
            positive_multiplier=best_params["positive_multiplier"]
        )

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=best_params["epochs"],
            batch_size=best_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        oof_prob.loc[inner_val_df.index] = val_prob

        del model
        cleanup_tf()

    if oof_prob.isna().any():
        raise ValueError("OOF probability 有缺值，請檢查 inner_cv 切分")

    return oof_prob

# =========================================================
# L. outer evaluation（只跑 M6）
# =========================================================
def evaluate_ft_transformer_m6(df, y, groups, selected_cols):
    fold_metrics = []
    best_params_per_fold = []

    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]

    # 強化版：適度擴大，不要大爆搜
    param_grid = {
        "d_token": [16, 32],
        "num_blocks": [2, 3],
        "num_heads": [4],
        "ff_dim": [64, 128],
        "dropout_rate": [0.1, 0.2, 0.3],
        "feature_dropout_rate": [0.0, 0.05],
        "mlp_hidden": [16, 32],
        "learning_rate": [1e-3],
        "l2_reg": [1e-5, 1e-4],
        "batch_size": [16, 32],
        "epochs": [50],
        "positive_multiplier": [1.5, 2.0]
    }

    param_list = list(itertools.product(
        param_grid["d_token"],
        param_grid["num_blocks"],
        param_grid["num_heads"],
        param_grid["ff_dim"],
        param_grid["dropout_rate"],
        param_grid["feature_dropout_rate"],
        param_grid["mlp_hidden"],
        param_grid["learning_rate"],
        param_grid["l2_reg"],
        param_grid["batch_size"],
        param_grid["epochs"],
        param_grid["positive_multiplier"]
    ))

    param_dicts = [
        {
            "d_token": p[0],
            "num_blocks": p[1],
            "num_heads": p[2],
            "ff_dim": p[3],
            "dropout_rate": p[4],
            "feature_dropout_rate": p[5],
            "mlp_hidden": p[6],
            "learning_rate": p[7],
            "l2_reg": p[8],
            "batch_size": p[9],
            "epochs": p[10],
            "positive_multiplier": p[11]
        }
        for p in param_list
    ]

    print("\n" + "=" * 100)
    print("Running Enhanced FT-Transformer on M6 only")
    print("=" * 100)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[M6] Outer Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_train_df = train_df[selected_cols].copy()
        X_test_df = test_df[selected_cols].copy()

        # 1) inner nested validation: 選最佳參數
        best_score = -1.0
        best_params = None

        for params in param_dicts:
            mean_inner_f1 = evaluate_param_on_inner_cv(
                X_df=X_train_df,
                y=y_train,
                groups=groups_train,
                cont_cols=cont_cols,
                lex_cols=lex_cols,
                params=params
            )

            if mean_inner_f1 > best_score:
                best_score = mean_inner_f1
                best_params = params.copy()

        best_params_per_fold.append(best_params)
        print(f"[M6] Fold {fold_idx} Best Params: {best_params}")
        print(f"[M6] Fold {fold_idx} Best Inner F1 (0.5 threshold): {best_score:.4f}")

        # 2) outer-train 做 OOF thresholding
        train_oof_prob = get_oof_probs_for_threshold(
            X_df=X_train_df,
            y=y_train,
            groups=groups_train,
            cont_cols=cont_cols,
            lex_cols=lex_cols,
            best_params=best_params
        )

        best_threshold, best_oof_f1 = find_best_threshold(
            y_true=y_train.loc[train_oof_prob.index].values,
            y_prob=train_oof_prob.values
        )

        # 3) 用最佳參數重訓 outer train，評估 outer test
        #    強化版：final train/val split 改成 group-aware
        tr_sub_idx, val_sub_idx = get_group_aware_train_val_split(
            X_train_df,
            y_train,
            groups_train,
            final_val_cv
        )

        X_tr_df = X_train_df.iloc[tr_sub_idx].copy()
        X_val_df = X_train_df.iloc[val_sub_idx].copy()

        y_tr = y_train.iloc[tr_sub_idx].values
        y_val = y_train.iloc[val_sub_idx].values

        X_tr, X_val, X_test = preprocess_train_valid_test(
            train_df=X_tr_df,
            valid_df=X_val_df,
            test_df=X_test_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_tr.shape[1],
            d_token=best_params["d_token"],
            num_blocks=best_params["num_blocks"],
            num_heads=best_params["num_heads"],
            ff_dim=best_params["ff_dim"],
            dropout_rate=best_params["dropout_rate"],
            feature_dropout_rate=best_params["feature_dropout_rate"],
            mlp_hidden=best_params["mlp_hidden"],
            learning_rate=best_params["learning_rate"],
            l2_reg=best_params["l2_reg"]
        )

        class_weight = make_class_weight(
            y_tr,
            positive_multiplier=best_params["positive_multiplier"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        model.fit(
            X_tr,
            y_tr,
            validation_data=(X_val, y_val),
            epochs=best_params["epochs"],
            batch_size=best_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob),
            "best_threshold": best_threshold,
            "oof_best_f1": best_oof_f1
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[M6] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f} | "
            f"PR_AUC={fold_result['average_precision']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "FT-Transformer (enhanced)",
        "Feature_Set": "M6: Semantic + Lexical + Financial",
        "Config": str(best_params_per_fold),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(ddof=1),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(ddof=1),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(ddof=1),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(ddof=1),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(ddof=1),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(ddof=1),

        "Mean_Best_Threshold": metrics_df["best_threshold"].mean(),
        "Threshold_std": metrics_df["best_threshold"].std(ddof=1),

        "Mean_OOF_Best_F1": metrics_df["oof_best_f1"].mean(),
        "OOF_Best_F1_std": metrics_df["oof_best_f1"].std(ddof=1)
    }

    return summary, metrics_df

# =========================================================
# M. 執行
# =========================================================
summary_result, fold_result_df = evaluate_ft_transformer_m6(
    df=df,
    y=y,
    groups=groups_all,
    selected_cols=m6_cols
)

results_df = pd.DataFrame([summary_result])
folds_df = fold_result_df.copy()

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== Summary Results =====")
print(results_df)

results_df.to_csv(
    "llama_ft_transformer_enhanced_M6_nested_groupcv_oof_threshold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "llama_ft_transformer_enhanced_M6_nested_groupcv_oof_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n結果已輸出：")
print("1. llama_ft_transformer_enhanced_M6_nested_groupcv_oof_threshold_results.csv")
print("2. llama_ft_transformer_enhanced_M6_nested_groupcv_oof_threshold_fold_results.csv")

I0000 00:00:1774944930.265871 1482750 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774944930.267278 1482750 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774944930.311563 1482750 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774944931.565674 1482750 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE


Running Enhanced FT-Transformer on M6 only

[M6] Outer Fold 1 started


E0000 00:00:1774944932.754196 1482750 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
E0000 00:00:1774944936.163590 1482750 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


In [2]:
import os
import gc
import random
import warnings
import itertools

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score
)
from tensorflow.keras import regularizers

# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")

tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)

# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# =========================================================
# C. 反轉反向指標（LLaMA）
# 分數越高 -> 越漂綠 / 風險越高
# =========================================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]

# =========================================================
# E. M6 特徵欄位（LLaMA）
# =========================================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

m6_cols = semantic_cols + lexical_cols + financial_cols

missing_cols = [c for c in (m6_cols + ["label", "Company"]) if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =========================================================
# F. CV 設定
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
final_val_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 999)

# =========================================================
# G. 前處理
# =========================================================
def preprocess_train_valid_test(train_df, valid_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_valid_cont = cont_imputer.transform(valid_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_valid_cont = scaler.transform(X_valid_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_cont = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_valid_lex = lex_imputer.transform(valid_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_lex = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_valid = np.concatenate([X_valid_cont, X_valid_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_valid, X_test


def preprocess_train_test_only(train_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_test

# =========================================================
# H. 小工具
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


def fast_predict(model, X, batch_size=256):
    X = np.asarray(X, dtype=np.float32)
    return model.predict(X, batch_size=batch_size, verbose=0).ravel()


def find_best_threshold(y_true, y_prob, thresholds=None):
    # 放寬搜尋範圍，不硬鎖 0.20~0.60
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19)

    best_threshold = 0.50
    best_score = -1.0

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = th

    return float(best_threshold), float(best_score)


def make_class_weight(y_train, positive_multiplier=1.2):
    counts = np.bincount(y_train)
    total = len(y_train)
    n_classes = len(np.unique(y_train))

    class_weight = {}
    for cls in np.unique(y_train):
        class_weight[int(cls)] = total / (n_classes * counts[cls])

    if 1 in class_weight:
        class_weight[1] *= positive_multiplier

    return class_weight


def get_group_aware_train_val_split(X_df, y_series, groups_series, splitter):
    for tr_idx, val_idx in splitter.split(X_df, y_series, groups=groups_series):
        return tr_idx, val_idx
    raise ValueError("無法建立 final train/val split")

# =========================================================
# I. 輕量版 FT-Transformer 元件
# =========================================================
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    """
    每個 numerical / binary feature 視為一個 token
    x: (batch, num_features)
    output: (batch, num_features, d_token)
    """
    def __init__(self, num_features, d_token, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.d_token = d_token

    def build(self, input_shape):
        self.weight = self.add_weight(
            name="feature_weight",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        self.bias = self.add_weight(
            name="feature_bias",
            shape=(self.num_features, self.d_token),
            initializer="zeros",
            trainable=True
        )
        self.col_embedding = self.add_weight(
            name="column_embedding",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.expand_dims(x, axis=-1)       # (B, F, 1)
        tokens = x * self.weight + self.bias # (B, F, D)
        tokens = tokens + self.col_embedding
        return tokens


class CLSLayer(tf.keras.layers.Layer):
    def __init__(self, d_token, **kwargs):
        super().__init__(**kwargs)
        self.d_token = d_token

    def build(self, input_shape):
        self.cls = self.add_weight(
            name="cls_token",
            shape=(1, 1, self.d_token),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls = tf.repeat(self.cls, repeats=batch_size, axis=0)
        return tf.concat([cls, x], axis=1)


def transformer_block(x, d_token, num_heads, ff_dim, dropout_rate, l2_reg):
    h = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_token // num_heads),
        dropout=dropout_rate
    )(h, h)
    h = tf.keras.layers.Dropout(dropout_rate)(h)
    x = tf.keras.layers.Add()([x, h])

    h2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h2 = tf.keras.layers.Dense(
        ff_dim,
        activation="gelu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    h2 = tf.keras.layers.Dense(
        d_token,
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    x = tf.keras.layers.Add()([x, h2])

    return x


def build_ft_transformer(
    num_features,
    d_token=16,
    num_blocks=1,
    num_heads=4,
    ff_dim=32,
    dropout_rate=0.3,
    feature_dropout_rate=0.0,
    mlp_hidden=16,
    learning_rate=1e-3,
    l2_reg=1e-4
):
    inputs = tf.keras.Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    x0 = tf.keras.layers.Dropout(feature_dropout_rate)(inputs)

    x = NumericalFeatureTokenizer(num_features=num_features, d_token=d_token)(x0)
    x = CLSLayer(d_token=d_token)(x)

    for _ in range(num_blocks):
        x = transformer_block(
            x=x,
            d_token=d_token,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate,
            l2_reg=l2_reg
        )

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    cls_token = x[:, 0, :]

    x = tf.keras.layers.Dense(
        mlp_hidden,
        activation="gelu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(cls_token)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# =========================================================
# J. inner CV 評估單一參數組
# 改成用 PR-AUC 做模型選擇
# =========================================================
def evaluate_param_on_inner_cv(X_df, y, groups, cont_cols, lex_cols, params):
    inner_scores = []

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            d_token=params["d_token"],
            num_blocks=params["num_blocks"],
            num_heads=params["num_heads"],
            ff_dim=params["ff_dim"],
            dropout_rate=params["dropout_rate"],
            feature_dropout_rate=params["feature_dropout_rate"],
            mlp_hidden=params["mlp_hidden"],
            learning_rate=params["learning_rate"],
            l2_reg=params["l2_reg"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        class_weight = make_class_weight(
            y_inner_train.values,
            positive_multiplier=1.2
        )

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        val_pr_auc = average_precision_score(y_inner_val, val_prob)
        inner_scores.append(val_pr_auc)

        del model
        cleanup_tf()

    return float(np.mean(inner_scores))

# =========================================================
# K. 用最佳參數在 outer-train 內做 OOF thresholding
# =========================================================
def get_oof_probs_for_threshold(X_df, y, groups, cont_cols, lex_cols, best_params):
    oof_prob = pd.Series(index=X_df.index, dtype=float)

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            d_token=best_params["d_token"],
            num_blocks=best_params["num_blocks"],
            num_heads=best_params["num_heads"],
            ff_dim=best_params["ff_dim"],
            dropout_rate=best_params["dropout_rate"],
            feature_dropout_rate=best_params["feature_dropout_rate"],
            mlp_hidden=best_params["mlp_hidden"],
            learning_rate=best_params["learning_rate"],
            l2_reg=best_params["l2_reg"]
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        class_weight = make_class_weight(
            y_inner_train.values,
            positive_multiplier=1.2
        )

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=best_params["epochs"],
            batch_size=best_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        oof_prob.loc[inner_val_df.index] = val_prob

        del model
        cleanup_tf()

    if oof_prob.isna().any():
        raise ValueError("OOF probability 有缺值，請檢查 inner_cv 切分")

    return oof_prob

# =========================================================
# L. outer evaluation（只跑 M6）
# =========================================================
def evaluate_ft_transformer_m6(df, y, groups, selected_cols):
    fold_metrics = []
    best_params_per_fold = []

    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]

    # 輕量穩定版 search space
    param_grid = {
        "d_token": [16],
        "num_blocks": [1, 2],
        "num_heads": [4],
        "ff_dim": [32, 64],
        "dropout_rate": [0.2, 0.3],
        "feature_dropout_rate": [0.0],
        "mlp_hidden": [16],
        "learning_rate": [1e-3],
        "l2_reg": [1e-4],
        "batch_size": [32],
        "epochs": [25, 30]
    }

    param_list = list(itertools.product(
        param_grid["d_token"],
        param_grid["num_blocks"],
        param_grid["num_heads"],
        param_grid["ff_dim"],
        param_grid["dropout_rate"],
        param_grid["feature_dropout_rate"],
        param_grid["mlp_hidden"],
        param_grid["learning_rate"],
        param_grid["l2_reg"],
        param_grid["batch_size"],
        param_grid["epochs"]
    ))

    param_dicts = [
        {
            "d_token": p[0],
            "num_blocks": p[1],
            "num_heads": p[2],
            "ff_dim": p[3],
            "dropout_rate": p[4],
            "feature_dropout_rate": p[5],
            "mlp_hidden": p[6],
            "learning_rate": p[7],
            "l2_reg": p[8],
            "batch_size": p[9],
            "epochs": p[10]
        }
        for p in param_list
    ]

    print("\n" + "=" * 100)
    print("Running Lightweight FT-Transformer on M6 only")
    print("=" * 100)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[M6] Outer Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_train_df = train_df[selected_cols].copy()
        X_test_df = test_df[selected_cols].copy()

        # 1) inner nested validation: 選最佳參數（看 PR-AUC）
        best_score = -1.0
        best_params = None

        for params in param_dicts:
            mean_inner_score = evaluate_param_on_inner_cv(
                X_df=X_train_df,
                y=y_train,
                groups=groups_train,
                cont_cols=cont_cols,
                lex_cols=lex_cols,
                params=params
            )

            if mean_inner_score > best_score:
                best_score = mean_inner_score
                best_params = params.copy()

        best_params_per_fold.append(best_params)
        print(f"[M6] Fold {fold_idx} Best Params: {best_params}")
        print(f"[M6] Fold {fold_idx} Best Inner PR-AUC: {best_score:.4f}")

        # 2) outer-train 做 OOF thresholding
        train_oof_prob = get_oof_probs_for_threshold(
            X_df=X_train_df,
            y=y_train,
            groups=groups_train,
            cont_cols=cont_cols,
            lex_cols=lex_cols,
            best_params=best_params
        )

        best_threshold, best_oof_f1 = find_best_threshold(
            y_true=y_train.loc[train_oof_prob.index].values,
            y_prob=train_oof_prob.values
        )

        # 3) 用最佳參數重訓 outer train，評估 outer test
        tr_sub_idx, val_sub_idx = get_group_aware_train_val_split(
            X_train_df,
            y_train,
            groups_train,
            final_val_cv
        )

        X_tr_df = X_train_df.iloc[tr_sub_idx].copy()
        X_val_df = X_train_df.iloc[val_sub_idx].copy()

        y_tr = y_train.iloc[tr_sub_idx].values
        y_val = y_train.iloc[val_sub_idx].values

        X_tr, X_val, X_test = preprocess_train_valid_test(
            train_df=X_tr_df,
            valid_df=X_val_df,
            test_df=X_test_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_tr.shape[1],
            d_token=best_params["d_token"],
            num_blocks=best_params["num_blocks"],
            num_heads=best_params["num_heads"],
            ff_dim=best_params["ff_dim"],
            dropout_rate=best_params["dropout_rate"],
            feature_dropout_rate=best_params["feature_dropout_rate"],
            mlp_hidden=best_params["mlp_hidden"],
            learning_rate=best_params["learning_rate"],
            l2_reg=best_params["l2_reg"]
        )

        class_weight = make_class_weight(
            y_tr,
            positive_multiplier=1.2
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        model.fit(
            X_tr,
            y_tr,
            validation_data=(X_val, y_val),
            epochs=best_params["epochs"],
            batch_size=best_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob),
            "best_threshold": best_threshold,
            "oof_best_f1": best_oof_f1
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[M6] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f} | "
            f"PR_AUC={fold_result['average_precision']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "FT-Transformer (lightweight)",
        "Feature_Set": "M6: Semantic + Lexical + Financial",
        "Config": str(best_params_per_fold),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(ddof=1),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(ddof=1),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(ddof=1),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(ddof=1),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(ddof=1),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(ddof=1),

        "Mean_Best_Threshold": metrics_df["best_threshold"].mean(),
        "Threshold_std": metrics_df["best_threshold"].std(ddof=1),

        "Mean_OOF_Best_F1": metrics_df["oof_best_f1"].mean(),
        "OOF_Best_F1_std": metrics_df["oof_best_f1"].std(ddof=1)
    }

    return summary, metrics_df

# =========================================================
# M. 執行
# =========================================================
summary_result, fold_result_df = evaluate_ft_transformer_m6(
    df=df,
    y=y,
    groups=groups_all,
    selected_cols=m6_cols
)

results_df = pd.DataFrame([summary_result])
folds_df = fold_result_df.copy()

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== Summary Results =====")
print(results_df)

results_df.to_csv(
    "llama_ft_transformer_lightweight_M6_nested_groupcv_oof_threshold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "llama_ft_transformer_lightweight_M6_nested_groupcv_oof_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n結果已輸出：")
print("1. llama_ft_transformer_lightweight_M6_nested_groupcv_oof_threshold_results.csv")
print("2. llama_ft_transformer_lightweight_M6_nested_groupcv_oof_threshold_fold_results.csv")


Running Lightweight FT-Transformer on M6 only

[M6] Outer Fold 1 started


E0000 00:00:1774958803.498284 2420886 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
E0000 00:00:1774958804.948778 2420886 util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


[M6] Fold 1 Best Params: {'d_token': 16, 'num_blocks': 1, 'num_heads': 4, 'ff_dim': 32, 'dropout_rate': 0.3, 'feature_dropout_rate': 0.0, 'mlp_hidden': 16, 'learning_rate': 0.001, 'l2_reg': 0.0001, 'batch_size': 32, 'epochs': 30}
[M6] Fold 1 Best Inner PR-AUC: 0.5873
[M6] Fold 1 done | Threshold=0.50 | F1=0.4000 | ROC_AUC=0.9477 | PR_AUC=0.9010

[M6] Outer Fold 2 started
[M6] Fold 2 Best Params: {'d_token': 16, 'num_blocks': 1, 'num_heads': 4, 'ff_dim': 64, 'dropout_rate': 0.2, 'feature_dropout_rate': 0.0, 'mlp_hidden': 16, 'learning_rate': 0.001, 'l2_reg': 0.0001, 'batch_size': 32, 'epochs': 30}
[M6] Fold 2 Best Inner PR-AUC: 0.7169
[M6] Fold 2 done | Threshold=0.65 | F1=0.6667 | ROC_AUC=0.9673 | PR_AUC=0.8571

[M6] Outer Fold 3 started
[M6] Fold 3 Best Params: {'d_token': 16, 'num_blocks': 1, 'num_heads': 4, 'ff_dim': 64, 'dropout_rate': 0.2, 'feature_dropout_rate': 0.0, 'mlp_hidden': 16, 'learning_rate': 0.001, 'l2_reg': 0.0001, 'batch_size': 32, 'epochs': 30}
[M6] Fold 3 Best Inner

In [3]:
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score
)
from tensorflow.keras import regularizers

# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")

tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)

# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# =========================================================
# C. 反轉反向指標（LLaMA）
# 分數越高 -> 越漂綠 / 風險越高
# =========================================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]

# =========================================================
# E. 特徵欄位（LLaMA）
# =========================================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# M4 = Semantic + Lexical
m4_cols = semantic_cols + lexical_cols

# M6 = Semantic + Lexical + Financial
m6_cols = semantic_cols + lexical_cols + financial_cols

required_cols = list(set(m4_cols + m6_cols + ["label", "Company"]))
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =========================================================
# F. CV 設定
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
final_val_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 999)

# =========================================================
# G. 固定參數（穩定快速版）
# =========================================================
FIXED_PARAMS = {
    "d_token": 16,
    "num_blocks": 1,
    "num_heads": 4,
    "ff_dim": 64,
    "dropout_rate": 0.2,
    "feature_dropout_rate": 0.0,
    "mlp_hidden": 16,
    "learning_rate": 1e-3,
    "l2_reg": 1e-4,
    "batch_size": 32,
    "epochs": 25
}

# =========================================================
# H. 前處理
# =========================================================
def preprocess_train_valid_test(train_df, valid_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_valid_cont = cont_imputer.transform(valid_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_valid_cont = scaler.transform(X_valid_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_cont = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_valid_lex = lex_imputer.transform(valid_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_lex = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_valid = np.concatenate([X_valid_cont, X_valid_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_valid, X_test


# =========================================================
# I. 小工具
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


def fast_predict(model, X, batch_size=256):
    X = np.asarray(X, dtype=np.float32)
    return model.predict(X, batch_size=batch_size, verbose=0).ravel()


def find_best_threshold(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19)

    best_threshold = 0.50
    best_score = -1.0

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = th

    return float(best_threshold), float(best_score)


def make_class_weight(y_train, positive_multiplier=1.2):
    counts = np.bincount(y_train)
    total = len(y_train)
    n_classes = len(np.unique(y_train))

    class_weight = {}
    for cls in np.unique(y_train):
        class_weight[int(cls)] = total / (n_classes * counts[cls])

    if 1 in class_weight:
        class_weight[1] *= positive_multiplier

    return class_weight


def get_group_aware_train_val_split(X_df, y_series, groups_series, splitter):
    for tr_idx, val_idx in splitter.split(X_df, y_series, groups=groups_series):
        return tr_idx, val_idx
    raise ValueError("無法建立 final train/val split")


# =========================================================
# J. FT-Transformer 元件
# =========================================================
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, d_token, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.d_token = d_token

    def build(self, input_shape):
        self.weight = self.add_weight(
            name="feature_weight",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        self.bias = self.add_weight(
            name="feature_bias",
            shape=(self.num_features, self.d_token),
            initializer="zeros",
            trainable=True
        )
        self.col_embedding = self.add_weight(
            name="column_embedding",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.expand_dims(x, axis=-1)       # (B, F, 1)
        tokens = x * self.weight + self.bias # (B, F, D)
        tokens = tokens + self.col_embedding
        return tokens


class CLSLayer(tf.keras.layers.Layer):
    def __init__(self, d_token, **kwargs):
        super().__init__(**kwargs)
        self.d_token = d_token

    def build(self, input_shape):
        self.cls = self.add_weight(
            name="cls_token",
            shape=(1, 1, self.d_token),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls = tf.repeat(self.cls, repeats=batch_size, axis=0)
        return tf.concat([cls, x], axis=1)


def transformer_block(x, d_token, num_heads, ff_dim, dropout_rate, l2_reg):
    h = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_token // num_heads),
        dropout=dropout_rate
    )(h, h)
    h = tf.keras.layers.Dropout(dropout_rate)(h)
    x = tf.keras.layers.Add()([x, h])

    h2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h2 = tf.keras.layers.Dense(
        ff_dim,
        activation="gelu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    h2 = tf.keras.layers.Dense(
        d_token,
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    x = tf.keras.layers.Add()([x, h2])

    return x


def build_ft_transformer(num_features, params):
    inputs = tf.keras.Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    x0 = tf.keras.layers.Dropout(params["feature_dropout_rate"])(inputs)

    x = NumericalFeatureTokenizer(
        num_features=num_features,
        d_token=params["d_token"]
    )(x0)

    x = CLSLayer(d_token=params["d_token"])(x)

    for _ in range(params["num_blocks"]):
        x = transformer_block(
            x=x,
            d_token=params["d_token"],
            num_heads=params["num_heads"],
            ff_dim=params["ff_dim"],
            dropout_rate=params["dropout_rate"],
            l2_reg=params["l2_reg"]
        )

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    cls_token = x[:, 0, :]

    x = tf.keras.layers.Dense(
        params["mlp_hidden"],
        activation="gelu",
        kernel_regularizer=regularizers.l2(params["l2_reg"])
    )(cls_token)
    x = tf.keras.layers.Dropout(params["dropout_rate"])(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        kernel_regularizer=regularizers.l2(params["l2_reg"])
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# =========================================================
# K. 用固定參數在 outer-train 內做 OOF thresholding
# =========================================================
def get_oof_probs_for_threshold(X_df, y, groups, cont_cols, lex_cols, fixed_params):
    oof_prob = pd.Series(index=X_df.index, dtype=float)

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            params=fixed_params
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        class_weight = make_class_weight(
            y_inner_train.values,
            positive_multiplier=1.2
        )

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=fixed_params["epochs"],
            batch_size=fixed_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        oof_prob.loc[inner_val_df.index] = val_prob

        del model
        cleanup_tf()

    if oof_prob.isna().any():
        raise ValueError("OOF probability 有缺值，請檢查 inner_cv 切分")

    return oof_prob


# =========================================================
# L. 單一 feature set 評估
# =========================================================
def evaluate_ft_transformer_fixed(df, y, groups, selected_cols, feature_set_name):
    fold_metrics = []

    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]

    print("\n" + "=" * 100)
    print(f"Running Fixed-Param FT-Transformer on {feature_set_name}")
    print("=" * 100)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[{feature_set_name}] Outer Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_train_df = train_df[selected_cols].copy()
        X_test_df = test_df[selected_cols].copy()

        # 1) 用固定參數做 outer-train OOF thresholding
        train_oof_prob = get_oof_probs_for_threshold(
            X_df=X_train_df,
            y=y_train,
            groups=groups_train,
            cont_cols=cont_cols,
            lex_cols=lex_cols,
            fixed_params=FIXED_PARAMS
        )

        best_threshold, best_oof_f1 = find_best_threshold(
            y_true=y_train.loc[train_oof_prob.index].values,
            y_prob=train_oof_prob.values
        )

        # 2) final train/val split
        tr_sub_idx, val_sub_idx = get_group_aware_train_val_split(
            X_train_df,
            y_train,
            groups_train,
            final_val_cv
        )

        X_tr_df = X_train_df.iloc[tr_sub_idx].copy()
        X_val_df = X_train_df.iloc[val_sub_idx].copy()

        y_tr = y_train.iloc[tr_sub_idx].values
        y_val = y_train.iloc[val_sub_idx].values

        X_tr, X_val, X_test = preprocess_train_valid_test(
            train_df=X_tr_df,
            valid_df=X_val_df,
            test_df=X_test_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_tr.shape[1],
            params=FIXED_PARAMS
        )

        class_weight = make_class_weight(
            y_tr,
            positive_multiplier=1.2
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        model.fit(
            X_tr,
            y_tr,
            validation_data=(X_val, y_val),
            epochs=FIXED_PARAMS["epochs"],
            batch_size=FIXED_PARAMS["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "Feature_Set": feature_set_name,
            "fold": fold_idx,
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob),
            "best_threshold": best_threshold,
            "oof_best_f1": best_oof_f1
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[{feature_set_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f} | "
            f"PR_AUC={fold_result['average_precision']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "FT-Transformer (fixed-param)",
        "Feature_Set": feature_set_name,
        "Config": str(FIXED_PARAMS),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(ddof=1),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(ddof=1),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(ddof=1),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(ddof=1),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(ddof=1),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(ddof=1),

        "Mean_Best_Threshold": metrics_df["best_threshold"].mean(),
        "Threshold_std": metrics_df["best_threshold"].std(ddof=1),

        "Mean_OOF_Best_F1": metrics_df["oof_best_f1"].mean(),
        "OOF_Best_F1_std": metrics_df["oof_best_f1"].std(ddof=1)
    }

    return summary, metrics_df


# =========================================================
# M. 執行 M4 + M6
# =========================================================
all_summaries = []
all_folds = []

feature_sets = [
    ("M4: Semantic + Lexical", m4_cols),
    ("M6: Semantic + Lexical + Financial", m6_cols),
]

for feature_set_name, cols in feature_sets:
    summary_result, fold_result_df = evaluate_ft_transformer_fixed(
        df=df,
        y=y,
        groups=groups_all,
        selected_cols=cols,
        feature_set_name=feature_set_name
    )
    all_summaries.append(summary_result)
    all_folds.append(fold_result_df)

results_df = pd.DataFrame(all_summaries)
folds_df = pd.concat(all_folds, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== Summary Results =====")
print(results_df)

results_df.to_csv(
    "llama_ft_transformer_fixedparam_M4_M6_groupcv_oof_threshold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "llama_ft_transformer_fixedparam_M4_M6_groupcv_oof_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n結果已輸出：")
print("1. llama_ft_transformer_fixedparam_M4_M6_groupcv_oof_threshold_results.csv")
print("2. llama_ft_transformer_fixedparam_M4_M6_groupcv_oof_threshold_fold_results.csv")


Running Fixed-Param FT-Transformer on M4: Semantic + Lexical

[M4: Semantic + Lexical] Outer Fold 1 started
[M4: Semantic + Lexical] Fold 1 done | Threshold=0.60 | F1=0.0000 | ROC_AUC=0.6802 | PR_AUC=0.4606

[M4: Semantic + Lexical] Outer Fold 2 started
[M4: Semantic + Lexical] Fold 2 done | Threshold=0.85 | F1=0.0000 | ROC_AUC=0.8889 | PR_AUC=0.7490

[M4: Semantic + Lexical] Outer Fold 3 started
[M4: Semantic + Lexical] Fold 3 done | Threshold=0.50 | F1=0.6000 | ROC_AUC=0.9730 | PR_AUC=0.8333

[M4: Semantic + Lexical] Outer Fold 4 started
[M4: Semantic + Lexical] Fold 4 done | Threshold=0.50 | F1=0.2000 | ROC_AUC=0.8099 | PR_AUC=0.1250

[M4: Semantic + Lexical] Outer Fold 5 started
[M4: Semantic + Lexical] Fold 5 done | Threshold=0.85 | F1=0.3077 | ROC_AUC=0.8860 | PR_AUC=0.3979

Running Fixed-Param FT-Transformer on M6: Semantic + Lexical + Financial

[M6: Semantic + Lexical + Financial] Outer Fold 1 started
[M6: Semantic + Lexical + Financial] Fold 1 done | Threshold=0.40 | F1=0.64

In [4]:
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score
)
from tensorflow.keras import regularizers

# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")

tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)

# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# =========================================================
# C. 反轉反向指標（LLaMA）
# 分數越高 -> 越漂綠 / 風險越高
# =========================================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]

# =========================================================
# E. 特徵欄位（LLaMA）
# =========================================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =========================================================
# F. Feature Sets（M1 ~ M6）
# =========================================================
m1_cols = semantic_cols
m2_cols = lexical_cols
m3_cols = financial_cols
m4_cols = semantic_cols + lexical_cols
m5_cols = semantic_cols + financial_cols
m6_cols = semantic_cols + lexical_cols + financial_cols

required_cols = list(set(
    m1_cols + m2_cols + m3_cols + m4_cols + m5_cols + m6_cols + ["label", "Company"]
))
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =========================================================
# G. CV 設定
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
final_val_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 999)

# =========================================================
# H. 固定參數（穩定快速版）
# =========================================================
FIXED_PARAMS = {
    "d_token": 16,
    "num_blocks": 1,
    "num_heads": 4,
    "ff_dim": 64,
    "dropout_rate": 0.2,
    "feature_dropout_rate": 0.0,
    "mlp_hidden": 16,
    "learning_rate": 1e-3,
    "l2_reg": 1e-4,
    "batch_size": 32,
    "epochs": 25
}

# =========================================================
# I. 前處理
# =========================================================
def preprocess_train_valid_test(train_df, valid_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_valid_cont = cont_imputer.transform(valid_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_valid_cont = scaler.transform(X_valid_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_cont = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_valid_lex = lex_imputer.transform(valid_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_lex = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_valid = np.concatenate([X_valid_cont, X_valid_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_valid, X_test


# =========================================================
# J. 小工具
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


def fast_predict(model, X, batch_size=256):
    X = np.asarray(X, dtype=np.float32)
    return model.predict(X, batch_size=batch_size, verbose=0).ravel()


def find_best_threshold(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19)

    best_threshold = 0.50
    best_score = -1.0

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = th

    return float(best_threshold), float(best_score)


def make_class_weight(y_train, positive_multiplier=1.2):
    counts = np.bincount(y_train)
    total = len(y_train)
    n_classes = len(np.unique(y_train))

    class_weight = {}
    for cls in np.unique(y_train):
        class_weight[int(cls)] = total / (n_classes * counts[cls])

    if 1 in class_weight:
        class_weight[1] *= positive_multiplier

    return class_weight


def get_group_aware_train_val_split(X_df, y_series, groups_series, splitter):
    for tr_idx, val_idx in splitter.split(X_df, y_series, groups=groups_series):
        return tr_idx, val_idx
    raise ValueError("無法建立 final train/val split")


# =========================================================
# K. FT-Transformer 元件
# =========================================================
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, d_token, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.d_token = d_token

    def build(self, input_shape):
        self.weight = self.add_weight(
            name="feature_weight",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        self.bias = self.add_weight(
            name="feature_bias",
            shape=(self.num_features, self.d_token),
            initializer="zeros",
            trainable=True
        )
        self.col_embedding = self.add_weight(
            name="column_embedding",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.expand_dims(x, axis=-1)       # (B, F, 1)
        tokens = x * self.weight + self.bias # (B, F, D)
        tokens = tokens + self.col_embedding
        return tokens


class CLSLayer(tf.keras.layers.Layer):
    def __init__(self, d_token, **kwargs):
        super().__init__(**kwargs)
        self.d_token = d_token

    def build(self, input_shape):
        self.cls = self.add_weight(
            name="cls_token",
            shape=(1, 1, self.d_token),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls = tf.repeat(self.cls, repeats=batch_size, axis=0)
        return tf.concat([cls, x], axis=1)


def transformer_block(x, d_token, num_heads, ff_dim, dropout_rate, l2_reg):
    h = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_token // num_heads),
        dropout=dropout_rate
    )(h, h)
    h = tf.keras.layers.Dropout(dropout_rate)(h)
    x = tf.keras.layers.Add()([x, h])

    h2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h2 = tf.keras.layers.Dense(
        ff_dim,
        activation="gelu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    h2 = tf.keras.layers.Dense(
        d_token,
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    x = tf.keras.layers.Add()([x, h2])

    return x


def build_ft_transformer(num_features, params):
    inputs = tf.keras.Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    x0 = tf.keras.layers.Dropout(params["feature_dropout_rate"])(inputs)

    x = NumericalFeatureTokenizer(
        num_features=num_features,
        d_token=params["d_token"]
    )(x0)

    x = CLSLayer(d_token=params["d_token"])(x)

    for _ in range(params["num_blocks"]):
        x = transformer_block(
            x=x,
            d_token=params["d_token"],
            num_heads=params["num_heads"],
            ff_dim=params["ff_dim"],
            dropout_rate=params["dropout_rate"],
            l2_reg=params["l2_reg"]
        )

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    cls_token = x[:, 0, :]

    x = tf.keras.layers.Dense(
        params["mlp_hidden"],
        activation="gelu",
        kernel_regularizer=regularizers.l2(params["l2_reg"])
    )(cls_token)
    x = tf.keras.layers.Dropout(params["dropout_rate"])(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        kernel_regularizer=regularizers.l2(params["l2_reg"])
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# =========================================================
# L. 用固定參數在 outer-train 內做 OOF thresholding
# =========================================================
def get_oof_probs_for_threshold(X_df, y, groups, cont_cols, lex_cols, fixed_params):
    oof_prob = pd.Series(index=X_df.index, dtype=float)

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            params=fixed_params
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        class_weight = make_class_weight(
            y_inner_train.values,
            positive_multiplier=1.2
        )

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=fixed_params["epochs"],
            batch_size=fixed_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        oof_prob.loc[inner_val_df.index] = val_prob

        del model
        cleanup_tf()

    if oof_prob.isna().any():
        raise ValueError("OOF probability 有缺值，請檢查 inner_cv 切分")

    return oof_prob


# =========================================================
# M. 單一 feature set 評估
# =========================================================
def evaluate_ft_transformer_fixed(df, y, groups, selected_cols, feature_set_name):
    fold_metrics = []

    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]

    print("\n" + "=" * 100)
    print(f"Running Fixed-Param FT-Transformer on {feature_set_name}")
    print("=" * 100)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[{feature_set_name}] Outer Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_train_df = train_df[selected_cols].copy()
        X_test_df = test_df[selected_cols].copy()

        # 1) 用固定參數做 outer-train OOF thresholding
        train_oof_prob = get_oof_probs_for_threshold(
            X_df=X_train_df,
            y=y_train,
            groups=groups_train,
            cont_cols=cont_cols,
            lex_cols=lex_cols,
            fixed_params=FIXED_PARAMS
        )

        best_threshold, best_oof_f1 = find_best_threshold(
            y_true=y_train.loc[train_oof_prob.index].values,
            y_prob=train_oof_prob.values
        )

        # 2) final train/val split
        tr_sub_idx, val_sub_idx = get_group_aware_train_val_split(
            X_train_df,
            y_train,
            groups_train,
            final_val_cv
        )

        X_tr_df = X_train_df.iloc[tr_sub_idx].copy()
        X_val_df = X_train_df.iloc[val_sub_idx].copy()

        y_tr = y_train.iloc[tr_sub_idx].values
        y_val = y_train.iloc[val_sub_idx].values

        X_tr, X_val, X_test = preprocess_train_valid_test(
            train_df=X_tr_df,
            valid_df=X_val_df,
            test_df=X_test_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_tr.shape[1],
            params=FIXED_PARAMS
        )

        class_weight = make_class_weight(
            y_tr,
            positive_multiplier=1.2
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        model.fit(
            X_tr,
            y_tr,
            validation_data=(X_val, y_val),
            epochs=FIXED_PARAMS["epochs"],
            batch_size=FIXED_PARAMS["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "Feature_Set": feature_set_name,
            "fold": fold_idx,
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob),
            "best_threshold": best_threshold,
            "oof_best_f1": best_oof_f1
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[{feature_set_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f} | "
            f"PR_AUC={fold_result['average_precision']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "FT-Transformer (fixed-param)",
        "Feature_Set": feature_set_name,
        "Config": str(FIXED_PARAMS),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(ddof=1),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(ddof=1),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(ddof=1),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(ddof=1),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(ddof=1),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(ddof=1),

        "Mean_Best_Threshold": metrics_df["best_threshold"].mean(),
        "Threshold_std": metrics_df["best_threshold"].std(ddof=1),

        "Mean_OOF_Best_F1": metrics_df["oof_best_f1"].mean(),
        "OOF_Best_F1_std": metrics_df["oof_best_f1"].std(ddof=1)
    }

    return summary, metrics_df


# =========================================================
# N. 執行 M1 + M2 + M3 + M4 + M5 + M6
# =========================================================
all_summaries = []
all_folds = []

feature_sets = [
    ("M1: Semantic", m1_cols),
    ("M2: Lexical", m2_cols),
    ("M3: Financial", m3_cols),
    ("M4: Semantic + Lexical", m4_cols),
    ("M5: Semantic + Financial", m5_cols),
    ("M6: Semantic + Lexical + Financial", m6_cols),
]

for feature_set_name, cols in feature_sets:
    summary_result, fold_result_df = evaluate_ft_transformer_fixed(
        df=df,
        y=y,
        groups=groups_all,
        selected_cols=cols,
        feature_set_name=feature_set_name
    )
    all_summaries.append(summary_result)
    all_folds.append(fold_result_df)

results_df = pd.DataFrame(all_summaries)
folds_df = pd.concat(all_folds, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== Summary Results =====")
print(results_df)

results_df.to_csv(
    "llama_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "llama_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n結果已輸出：")
print("1. llama_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_results.csv")
print("2. llama_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_fold_results.csv")


Running Fixed-Param FT-Transformer on M1: Semantic

[M1: Semantic] Outer Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.50 | F1=0.6667 | ROC_AUC=0.9273 | PR_AUC=0.7990

[M1: Semantic] Outer Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.35 | F1=0.2182 | ROC_AUC=0.9722 | PR_AUC=0.8571

[M1: Semantic] Outer Fold 3 started
[M1: Semantic] Fold 3 done | Threshold=0.75 | F1=0.7500 | ROC_AUC=0.9966 | PR_AUC=0.9500

[M1: Semantic] Outer Fold 4 started
[M1: Semantic] Fold 4 done | Threshold=0.45 | F1=0.1538 | ROC_AUC=0.8592 | PR_AUC=0.1422

[M1: Semantic] Outer Fold 5 started
[M1: Semantic] Fold 5 done | Threshold=0.55 | F1=0.5833 | ROC_AUC=0.8750 | PR_AUC=0.3585

Running Fixed-Param FT-Transformer on M2: Lexical

[M2: Lexical] Outer Fold 1 started
[M2: Lexical] Fold 1 done | Threshold=0.35 | F1=0.5714 | ROC_AUC=0.8750 | PR_AUC=0.5697

[M2: Lexical] Outer Fold 2 started
[M2: Lexical] Fold 2 done | Threshold=0.55 | F1=0.0000 | ROC_AUC=0.9118 | PR_AUC=0.5000

[M2: Lexical] Oute

In [5]:
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score
)
from tensorflow.keras import regularizers

# =========================================================
# A. 基本設定
# =========================================================
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # 強制 CPU

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")

tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(2)

# =========================================================
# B. 讀資料
# =========================================================
DATA_PATH = "final_dataset_for_ml_FULL.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# =========================================================
# C. 反轉反向指標（LLaMA）
# 分數越高 -> 越漂綠 / 風險越高
# =========================================================
reverse_map = {
    "chatgpt_vagueness_score_1": "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1": "chatgpt_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =========================================================
# D. target / groups
# =========================================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]

# =========================================================
# E. 特徵欄位（LLaMA）
# =========================================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_risk_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_risk_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =========================================================
# F. Feature Sets（M1 ~ M6）
# =========================================================
m1_cols = semantic_cols
m2_cols = lexical_cols
m3_cols = financial_cols
m4_cols = semantic_cols + lexical_cols
m5_cols = semantic_cols + financial_cols
m6_cols = semantic_cols + lexical_cols + financial_cols

required_cols = list(set(
    m1_cols + m2_cols + m3_cols + m4_cols + m5_cols + m6_cols + ["label", "Company"]
))
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =========================================================
# G. CV 設定
# =========================================================
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
final_val_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 999)

# =========================================================
# H. 固定參數（穩定快速版）
# =========================================================
FIXED_PARAMS = {
    "d_token": 16,
    "num_blocks": 1,
    "num_heads": 4,
    "ff_dim": 64,
    "dropout_rate": 0.2,
    "feature_dropout_rate": 0.0,
    "mlp_hidden": 16,
    "learning_rate": 1e-3,
    "l2_reg": 1e-4,
    "batch_size": 32,
    "epochs": 25
}

# =========================================================
# I. 前處理
# =========================================================
def preprocess_train_valid_test(train_df, valid_df, test_df, cont_cols, lex_cols):
    if len(cont_cols) > 0:
        cont_imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_train_cont = cont_imputer.fit_transform(train_df[cont_cols])
        X_valid_cont = cont_imputer.transform(valid_df[cont_cols])
        X_test_cont = cont_imputer.transform(test_df[cont_cols])

        X_train_cont = scaler.fit_transform(X_train_cont)
        X_valid_cont = scaler.transform(X_valid_cont)
        X_test_cont = scaler.transform(X_test_cont)
    else:
        X_train_cont = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_cont = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_cont = np.zeros((len(test_df), 0), dtype=np.float32)

    if len(lex_cols) > 0:
        lex_imputer = SimpleImputer(strategy="most_frequent")

        X_train_lex = lex_imputer.fit_transform(train_df[lex_cols]).astype(np.float32)
        X_valid_lex = lex_imputer.transform(valid_df[lex_cols]).astype(np.float32)
        X_test_lex = lex_imputer.transform(test_df[lex_cols]).astype(np.float32)
    else:
        X_train_lex = np.zeros((len(train_df), 0), dtype=np.float32)
        X_valid_lex = np.zeros((len(valid_df), 0), dtype=np.float32)
        X_test_lex = np.zeros((len(test_df), 0), dtype=np.float32)

    X_train = np.concatenate([X_train_cont, X_train_lex], axis=1).astype(np.float32)
    X_valid = np.concatenate([X_valid_cont, X_valid_lex], axis=1).astype(np.float32)
    X_test = np.concatenate([X_test_cont, X_test_lex], axis=1).astype(np.float32)

    return X_train, X_valid, X_test


# =========================================================
# J. 小工具
# =========================================================
def cleanup_tf():
    tf.keras.backend.clear_session()
    gc.collect()


def fast_predict(model, X, batch_size=256):
    X = np.asarray(X, dtype=np.float32)
    return model.predict(X, batch_size=batch_size, verbose=0).ravel()


def find_best_threshold(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19)

    best_threshold = 0.50
    best_score = -1.0

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = th

    return float(best_threshold), float(best_score)


def make_class_weight(y_train, positive_multiplier=1.2):
    counts = np.bincount(y_train)
    total = len(y_train)
    n_classes = len(np.unique(y_train))

    class_weight = {}
    for cls in np.unique(y_train):
        class_weight[int(cls)] = total / (n_classes * counts[cls])

    if 1 in class_weight:
        class_weight[1] *= positive_multiplier

    return class_weight


def get_group_aware_train_val_split(X_df, y_series, groups_series, splitter):
    for tr_idx, val_idx in splitter.split(X_df, y_series, groups=groups_series):
        return tr_idx, val_idx
    raise ValueError("無法建立 final train/val split")


# =========================================================
# K. FT-Transformer 元件
# =========================================================
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, d_token, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.d_token = d_token

    def build(self, input_shape):
        self.weight = self.add_weight(
            name="feature_weight",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        self.bias = self.add_weight(
            name="feature_bias",
            shape=(self.num_features, self.d_token),
            initializer="zeros",
            trainable=True
        )
        self.col_embedding = self.add_weight(
            name="column_embedding",
            shape=(self.num_features, self.d_token),
            initializer="glorot_uniform",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.expand_dims(x, axis=-1)       # (B, F, 1)
        tokens = x * self.weight + self.bias # (B, F, D)
        tokens = tokens + self.col_embedding
        return tokens


class CLSLayer(tf.keras.layers.Layer):
    def __init__(self, d_token, **kwargs):
        super().__init__(**kwargs)
        self.d_token = d_token

    def build(self, input_shape):
        self.cls = self.add_weight(
            name="cls_token",
            shape=(1, 1, self.d_token),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls = tf.repeat(self.cls, repeats=batch_size, axis=0)
        return tf.concat([cls, x], axis=1)


def transformer_block(x, d_token, num_heads, ff_dim, dropout_rate, l2_reg):
    h = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=max(1, d_token // num_heads),
        dropout=dropout_rate
    )(h, h)
    h = tf.keras.layers.Dropout(dropout_rate)(h)
    x = tf.keras.layers.Add()([x, h])

    h2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    h2 = tf.keras.layers.Dense(
        ff_dim,
        activation="gelu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    h2 = tf.keras.layers.Dense(
        d_token,
        kernel_regularizer=regularizers.l2(l2_reg)
    )(h2)
    h2 = tf.keras.layers.Dropout(dropout_rate)(h2)
    x = tf.keras.layers.Add()([x, h2])

    return x


def build_ft_transformer(num_features, params):
    inputs = tf.keras.Input(shape=(num_features,), dtype=tf.float32, name="feature_input")

    x0 = tf.keras.layers.Dropout(params["feature_dropout_rate"])(inputs)

    x = NumericalFeatureTokenizer(
        num_features=num_features,
        d_token=params["d_token"]
    )(x0)

    x = CLSLayer(d_token=params["d_token"])(x)

    for _ in range(params["num_blocks"]):
        x = transformer_block(
            x=x,
            d_token=params["d_token"],
            num_heads=params["num_heads"],
            ff_dim=params["ff_dim"],
            dropout_rate=params["dropout_rate"],
            l2_reg=params["l2_reg"]
        )

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    cls_token = x[:, 0, :]

    x = tf.keras.layers.Dense(
        params["mlp_hidden"],
        activation="gelu",
        kernel_regularizer=regularizers.l2(params["l2_reg"])
    )(cls_token)
    x = tf.keras.layers.Dropout(params["dropout_rate"])(x)

    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        kernel_regularizer=regularizers.l2(params["l2_reg"])
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# =========================================================
# L. 用固定參數在 outer-train 內做 OOF thresholding
# =========================================================
def get_oof_probs_for_threshold(X_df, y, groups, cont_cols, lex_cols, fixed_params):
    oof_prob = pd.Series(index=X_df.index, dtype=float)

    for inner_train_idx, inner_val_idx in inner_cv.split(X_df, y, groups=groups):
        inner_train_df = X_df.iloc[inner_train_idx].copy()
        inner_val_df = X_df.iloc[inner_val_idx].copy()

        y_inner_train = y.iloc[inner_train_idx].copy()
        y_inner_val = y.iloc[inner_val_idx].copy()

        X_inner_train, X_inner_val, _ = preprocess_train_valid_test(
            train_df=inner_train_df,
            valid_df=inner_val_df,
            test_df=inner_val_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_inner_train.shape[1],
            params=fixed_params
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        class_weight = make_class_weight(
            y_inner_train.values,
            positive_multiplier=1.2
        )

        model.fit(
            X_inner_train,
            y_inner_train.values,
            validation_data=(X_inner_val, y_inner_val.values),
            epochs=fixed_params["epochs"],
            batch_size=fixed_params["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_prob = fast_predict(model, X_inner_val)
        oof_prob.loc[inner_val_df.index] = val_prob

        del model
        cleanup_tf()

    if oof_prob.isna().any():
        raise ValueError("OOF probability 有缺值，請檢查 inner_cv 切分")

    return oof_prob


# =========================================================
# M. 單一 feature set 評估
# =========================================================
def evaluate_ft_transformer_fixed(df, y, groups, selected_cols, feature_set_name):
    fold_metrics = []

    cont_cols = [c for c in selected_cols if c in (semantic_cols + financial_cols)]
    lex_cols = [c for c in selected_cols if c in lexical_cols]

    print("\n" + "=" * 100)
    print(f"Running Fixed-Param FT-Transformer on {feature_set_name}")
    print("=" * 100)

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(df[selected_cols], y, groups=groups), start=1
    ):
        print(f"\n[{feature_set_name}] Outer Fold {fold_idx} started")

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        X_train_df = train_df[selected_cols].copy()
        X_test_df = test_df[selected_cols].copy()

        # 1) 用固定參數做 outer-train OOF thresholding
        train_oof_prob = get_oof_probs_for_threshold(
            X_df=X_train_df,
            y=y_train,
            groups=groups_train,
            cont_cols=cont_cols,
            lex_cols=lex_cols,
            fixed_params=FIXED_PARAMS
        )

        best_threshold, best_oof_f1 = find_best_threshold(
            y_true=y_train.loc[train_oof_prob.index].values,
            y_prob=train_oof_prob.values
        )

        # 2) final train/val split
        tr_sub_idx, val_sub_idx = get_group_aware_train_val_split(
            X_train_df,
            y_train,
            groups_train,
            final_val_cv
        )

        X_tr_df = X_train_df.iloc[tr_sub_idx].copy()
        X_val_df = X_train_df.iloc[val_sub_idx].copy()

        y_tr = y_train.iloc[tr_sub_idx].values
        y_val = y_train.iloc[val_sub_idx].values

        X_tr, X_val, X_test = preprocess_train_valid_test(
            train_df=X_tr_df,
            valid_df=X_val_df,
            test_df=X_test_df,
            cont_cols=cont_cols,
            lex_cols=lex_cols
        )

        cleanup_tf()

        model = build_ft_transformer(
            num_features=X_tr.shape[1],
            params=FIXED_PARAMS
        )

        class_weight = make_class_weight(
            y_tr,
            positive_multiplier=1.2
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            min_delta=1e-4
        )

        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=0
        )

        model.fit(
            X_tr,
            y_tr,
            validation_data=(X_val, y_val),
            epochs=FIXED_PARAMS["epochs"],
            batch_size=FIXED_PARAMS["batch_size"],
            class_weight=class_weight,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        test_prob = fast_predict(model, X_test)
        y_pred = (test_prob >= best_threshold).astype(int)

        try:
            roc_auc = roc_auc_score(y_test, test_prob)
        except ValueError:
            roc_auc = np.nan

        fold_result = {
            "Feature_Set": feature_set_name,
            "fold": fold_idx,
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob),
            "best_threshold": best_threshold,
            "oof_best_f1": best_oof_f1
        }
        fold_metrics.append(fold_result)

        del model
        cleanup_tf()

        print(
            f"[{feature_set_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f} | "
            f"PR_AUC={fold_result['average_precision']:.4f}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "FT-Transformer (fixed-param)",
        "Feature_Set": feature_set_name,
        "Config": str(FIXED_PARAMS),
        "Num_Features": len(selected_cols),
        "Num_Continuous": len(cont_cols),
        "Num_Lexical": len(lex_cols),

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(ddof=1),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(ddof=1),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(ddof=1),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(ddof=1),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(ddof=1),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(ddof=1),

        "Mean_Best_Threshold": metrics_df["best_threshold"].mean(),
        "Threshold_std": metrics_df["best_threshold"].std(ddof=1),

        "Mean_OOF_Best_F1": metrics_df["oof_best_f1"].mean(),
        "OOF_Best_F1_std": metrics_df["oof_best_f1"].std(ddof=1)
    }

    return summary, metrics_df


# =========================================================
# N. 執行 M1 + M2 + M3 + M4 + M5 + M6
# =========================================================
all_summaries = []
all_folds = []

feature_sets = [
    ("M1: Semantic", m1_cols),
    ("M2: Lexical", m2_cols),
    ("M3: Financial", m3_cols),
    ("M4: Semantic + Lexical", m4_cols),
    ("M5: Semantic + Financial", m5_cols),
    ("M6: Semantic + Lexical + Financial", m6_cols),
]

for feature_set_name, cols in feature_sets:
    summary_result, fold_result_df = evaluate_ft_transformer_fixed(
        df=df,
        y=y,
        groups=groups_all,
        selected_cols=cols,
        feature_set_name=feature_set_name
    )
    all_summaries.append(summary_result)
    all_folds.append(fold_result_df)

results_df = pd.DataFrame(all_summaries)
folds_df = pd.concat(all_folds, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== Summary Results =====")
print(results_df)

results_df.to_csv(
    "chatgpt_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "chatgpt_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n結果已輸出：")
print("1. chatgpt_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_results.csv")
print("2. chatgpt_ft_transformer_fixedparam_M1_M6_groupcv_oof_threshold_fold_results.csv")


Running Fixed-Param FT-Transformer on M1: Semantic

[M1: Semantic] Outer Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.65 | F1=0.8000 | ROC_AUC=0.9302 | PR_AUC=0.6978

[M1: Semantic] Outer Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.85 | F1=0.8333 | ROC_AUC=0.9935 | PR_AUC=0.9583

[M1: Semantic] Outer Fold 3 started
[M1: Semantic] Fold 3 done | Threshold=0.75 | F1=0.4000 | ROC_AUC=0.9426 | PR_AUC=0.5682

[M1: Semantic] Outer Fold 4 started
[M1: Semantic] Fold 4 done | Threshold=0.60 | F1=0.2667 | ROC_AUC=0.9648 | PR_AUC=0.6429

[M1: Semantic] Outer Fold 5 started
[M1: Semantic] Fold 5 done | Threshold=0.45 | F1=0.7273 | ROC_AUC=0.9079 | PR_AUC=0.4269

Running Fixed-Param FT-Transformer on M2: Lexical

[M2: Lexical] Outer Fold 1 started
[M2: Lexical] Fold 1 done | Threshold=0.35 | F1=0.5714 | ROC_AUC=0.8750 | PR_AUC=0.5697

[M2: Lexical] Outer Fold 2 started
[M2: Lexical] Fold 2 done | Threshold=0.55 | F1=0.0000 | ROC_AUC=0.9118 | PR_AUC=0.5000

[M2: Lexical] Oute